# Modeling

Packages and setup

In [ ]:
# Imports & settings
from foodcast.imports import *
notebook_settings() 
os.chdir(PROJECT_ROOT)
BASE_DIR, DATA_DIR_1, DATA_DIR_2, DATA_DIR_3, DATA_DIR_4, DATA_DIR_3_x = return_dir()
DATA_DIR_3_1, DATA_DIR_3_2, DATA_DIR_3_3, DATA_DIR_3_4, DATA_DIR_3_5, DATA_DIR_3_6, DATA_DIR_3_7, DATA_DIR_3_8, _ = DATA_DIR_3_x
from foodcast.tools.rolling import rolling_window_avg, add_interday_variables, add_intraday_variables, unroll, season_from_month
from foodcast.tools.takeout import takeout

# Load Data
location_ids_by_coverage = load_loc_ids()
locations, before_after_details_true, items_tagged, customers = load_static()
data = load_all_res_3_7_truly_consolidated()

# Load special data
customers_before_after = pd.read_pickle(DATA_DIR_3 / 'customers_before_after.pkl')
#animal_categories_items = pd.read_pickle(DATA_DIR_3 / 'animal_categories_items.pkl')

In [ ]:
animal_product_categories = [
    'lamb','chunked_beef_or_pork','pulled_pork', # A_1
    'beef_or_pork_burger','ground_meat','meatballs', # A_2
    'sausage','bacon','breakfast_sausage_patty', # A_3
    'unfried_chicken','fried_chicken', # A_4
    'savory_dairy','sweet_dairy', # A_5
    'egg' # No MPBAs
]

category_mappings = {
    'lamb': ('|'.join(['lamb','sheep','mutton']),
                '|'.join(['-v','v-','^v ',' v$','vegan','black sheep'])),
    'beef_or_pork_burger': ('|'.join(['burg','patty']),
                            '|'.join(['-v','v-','^v ',' v$','vegan','impossible','beyond'])),
    'sausage': ('|'.join(['sausage','brat','link','kielbasa']),
                '|'.join(['-v','v-','^v ',' v$','vegan','beyond','impossible'])),
    'meatballs': ('|'.join(['meatball','kofta']),
                    '|'.join(['-v','v-','^v ',' v$','vegan','impossible','beyond'])),
    'bacon': ('|'.join(['bacon','pancetta','prosciutto']),
                '|'.join(['-v','v-','^v ',' v$','vegan','thrilling','bakon','vegan bacon'])),
    'ground_meat': ('|'.join(['ground pork','ground beef', 'ground meat', 'ground chicken', 'ground lamb','mince']),
                    '|'.join(['-v','v-','^v ',' v$','vegan','impossible','beyond','tofu','seitan'])),
    'breakfast_sausage_patty': ('|'.join(['sausage patty','breakfast sausage','breakfast patty']),
                                '|'.join(['-v','v-','^v ',' v$','vegan','impossible','beyond'])),
    'chunked_beef_or_pork': ('|'.join(['beef chunks','carnitas']),
                                '|'.join(['-v','v-','^v ',' v$','vegan','tofu','seitan'])),
    'pulled_pork': ('|'.join(['pulled','pork']),
                    '|'.join(['-v','v-','^v ',' v$','vegan','jackfruit'])),
    'unfried_chicken': ('|'.join(['chicken','turkey']),
                        '|'.join(['-v','v-','^v ',' v$','vegan','unchicken','un\'chicken'])),
    'fried_chicken': ('|'.join(['fried chicken','wing','tender','nugget','drumstick','popcorn chicken']),
                        '|'.join(['-v','v-','^v ',' v$','vegan','tofu','seitan'])),
    'savory_dairy': ('|'.join(['cheese','chz','cream','yogurt','aioli',
                                'butter','feta','mozzarella','cheddar',
                                'parmesan','queso','ricotta']),
                        '|'.join(['-v','v-','^v ',' v$','vegan','dairy-free','df','non-dairy','dairy free'])),
    'sweet_dairy': ('|'.join(['dessert','cake','custard','cream','cheescake','pudding',]),
                    '|'.join(['-v','v-','^v ',' v$','vegan','dairy-free','df','non-dairy','dairy free'])),
    'egg': ('|'.join(['egg','mayo','aioli','omelet','omelette','scramble','deviled']),
            '|'.join(['-v','v-','^v ',' v$','vegan','just egg']))
}

for loc_id in location_ids_by_coverage:
    
    print(loc_id)
    
    if loc_id in [
        'VLZX7K2M9QD4T','SRQS8F7JWA9MZ','2HRX9P6HKXA8V','JHDN7CF1C03X5',
        'L69HYJ4Y3TR91','ED5J990H5VAZT','W8T41JZK0ZMEP']:
        dish_labels = pd.read_csv(Path('scripts') / 'labeling' / 'dish_labels' / f'{loc_id}.csv', index_col='item_name')
    elif loc_id in [
        "EMBVNVD207CC6","C0BE4NDSW26QN","75WYSXR9QBK5M","V3Q26BHF3SE2H","LBZEEFSBJNB3Z",
        "SAFK7ND1HR6XS","CB2KHY1C2G9PT","S8MT0YGD2KTN9","LFZFT3VASXPED","1SQPTEGYPH0GA",
        "9XKJD8DQTH559","LQ5EH4BKGV61T","78AY09MVJVTYE"]:
        dish_labels = pd.read_csv(Path('scripts') / 'labeling' / 'dish_labels_t2' / f'{loc_id}_1.csv', index_col='item_name')
    
    df = data[loc_id].join(dish_labels, on='item_name', lsuffix='', rsuffix='_dish')
    
    rename_map = {
        f"{col}_dish": col
        for col in animal_product_categories
        if f"{col}_dish" in df.columns
    }
    
    df = df.rename(columns=rename_map)
    
    for cat, (keywords, anti_keywords) in category_mappings.items():
        df[cat] = (
            df[cat]
            .mask(
                df["item_modifications"].str.contains(keywords, case=False, na=False),
                True)
            .mask(
                df["item_modifications"].str.contains(anti_keywords, case=False, na=False),
                False))

    print(df.columns)

    df = (
        df
        .set_index('created_at')
        .assign(meat = lambda df: ~df.vegetarian)
        .pipe(rolling_window_avg, 'vegan', 'item_price', 'item_quantity', 1, 'D')
        .pipe(rolling_window_avg, 'vegetarian', 'item_price', 'item_quantity', 1, 'D')
        .pipe(rolling_window_avg, 'meat', 'item_price', 'item_quantity', 1, 'D')
        .reset_index()
        .assign(created_at = lambda df: df.created_at.dt.tz_convert('utc')))

    data[loc_id] = df

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=pd.errors.PerformanceWarning)

    df_all_list = []   
    for loc_id in location_ids_by_coverage:
        df = data[loc_id]
        df_all_list.append(df)

    df_all = pd.concat(df_all_list, ignore_index=True)  
        
    del data
    gc.collect()

    model_dat = (
        df_all
        # Outcomes
        .assign( 
            vegan_outcome = lambda df: 1*df['vegan'],
            vegetarian_outcome = lambda df: 1*df['vegetarian'],
            total_outcome = 1,
            nonvegan_outcome = lambda df: 1 - df['vegan_outcome'],
            meat_outcome = lambda df: 1 - df['vegetarian_outcome'],
            chicken_fish_outcome = lambda df: 1*df['chicken_fish'])
        
        # Locations
        .astype({
            'location_id':'category'})
        
        # Time variables
        .set_index('created_at')
        .pipe(add_intraday_variables)
        .pipe(add_interday_variables)
        .pipe(lambda df: print(f'\nEntries after adding time variables: \n{df.shape[0]}') or df)
        
        # Rolling
        .pipe(unroll)
        .pipe(lambda df: print(f'\nEntries after unrolling multi item orders: \n{df.shape[0]}') or df))
    
    del df_all
    gc.collect()

In [ ]:
def make_exposure(df, location_id, cutoff, tz="UTC"):
    """
    Returns exposure indicator corresponding to location_id
    """
    idx = df.index

    dt = pd.to_datetime(cutoff)
    if dt.tzinfo is None: dt = dt.tz_localize(tz)
    else: dt = dt.tz_convert(tz)

    mask = (df["location_id"] == location_id) & (dt <= idx)
    return pd.Series(0, index=df.index).mask(mask, 1)

In [ ]:
def targeted_from_map(df, mapping, default=0, dtype="int8"):
    """
    mapping: list of (location_id_list, source_colname)
    Returns a Series that, for each row, uses the source column specified by
    the first matching location_id_list in mapping; otherwise default.
    """
    out = pd.Series(default, index=df.index)
    for locs, col in mapping:
        mask = df["location_id"].isin(locs)
        out.loc[mask] = df.loc[mask, col].astype(dtype).values
    return out.astype(dtype)   

import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=pd.errors.PerformanceWarning)  
    # ALL

    # 1
    breakfast_map = [
        (["2HRX9P6HKXA8V", "JHDN7CF1C03X5"], "sausage"),
        (["L69HYJ4Y3TR91"], "breakfast_sausage_patty"),
        (["ED5J990H5VAZT"], "bacon"),
    ]
    textured_map = [
        (["VLZX7K2M9QD4T"], "lamb"),
    ]
    untextured_map = [
        (["SRQS8F7JWA9MZ", "JHDN7CF1C03X5"], "beef_or_pork_burger"),
    ]

    # 2
    breakfast_map_2 = [
        (["2HRX9P6HKXA8V", "JHDN7CF1C03X5", "78AY09MVJVTYE"], "sausage"),
        (["L69HYJ4Y3TR91", "W8T41JZK0ZMEP", "V3Q26BHF3SE2H"], "breakfast_sausage_patty"),
        (["ED5J990H5VAZT"], "bacon"),
    ]
    textured_map_2 = [
        (["VLZX7K2M9QD4T"], "lamb"),
        (["SAFK7ND1HR6XS"], "pulled_pork")
    ]
    untextured_map_2 = [
        (["SRQS8F7JWA9MZ", 
        "JHDN7CF1C03X5", 
        "C0BE4NDSW26QN", 
        "S8MT0YGD2KTN9",
        "9XKJD8DQTH559",
        "LQ5EH4BKGV61T"], "beef_or_pork_burger"),
        (["1SQPTEGYPH0GA"], "meatballs")
    ]
    chicken_map_2 = [
        (["V3Q26BHF3SE2H"], "fried_chicken")
    ]
    dairy_map_2 = [
        (["W8T41JZK0ZMEP"], "sweet_dairy"),
        (["EMBVNVD207CC6", "9XKJD8DQTH559"], "savory_dairy"),
    ]

    # CUSTOMERS (true filtering occurs)

    # 1
    breakfast_map_c = [
        (["2HRX9P6HKXA8V"], "sausage"),
        (["L69HYJ4Y3TR91"], "breakfast_sausage_patty"),
        (["ED5J990H5VAZT"], "bacon"),
    ]
    untextured_map_c = [
        (["SRQS8F7JWA9MZ"], "beef_or_pork_burger"),
    ]

    # 2
    breakfast_map_2_c = [
        (["2HRX9P6HKXA8V", "78AY09MVJVTYE"], "sausage"),
        (["L69HYJ4Y3TR91", "W8T41JZK0ZMEP", "V3Q26BHF3SE2H"], "breakfast_sausage_patty"),
        (["ED5J990H5VAZT"], "bacon"),
    ]
    textured_map_2_c = [
        (["SAFK7ND1HR6XS"], "pulled_pork")
    ]
    untextured_map_2_c = [
        (["SRQS8F7JWA9MZ", 
        "C0BE4NDSW26QN", 
        "S8MT0YGD2KTN9",
        "9XKJD8DQTH559",
        "LQ5EH4BKGV61T"], "beef_or_pork_burger"),
        (["1SQPTEGYPH0GA"], "meatballs")
    ]
    chicken_map_2_c = [
        (["V3Q26BHF3SE2H"], "fried_chicken")
    ]
    dairy_map_2_c = [
        (["W8T41JZK0ZMEP"], "sweet_dairy"),
        (["EMBVNVD207CC6", "9XKJD8DQTH559"], "savory_dairy"),
    ]

    model_data = (
        model_dat
        .assign(
            breakfast=lambda df: targeted_from_map(df, breakfast_map),
            textured=lambda df: targeted_from_map(df, textured_map),
            untextured=lambda df: targeted_from_map(df, untextured_map),
            breakfast_t2=lambda df: targeted_from_map(df, breakfast_map_2),
            textured_t2=lambda df: targeted_from_map(df, textured_map_2),
            untextured_t2=lambda df: targeted_from_map(df, untextured_map_2),
            chicken_t2=lambda df: targeted_from_map(df, chicken_map_2),
            dairy_t2=lambda df: targeted_from_map(df, dairy_map_2),

            breakfast_c=lambda df: targeted_from_map(df, breakfast_map_c),
            textured_c=0,
            untextured_c=lambda df: targeted_from_map(df, untextured_map_c),
            breakfast_t2_c=lambda df: targeted_from_map(df, breakfast_map_2_c),
            textured_t2_c=lambda df: targeted_from_map(df, textured_map_2_c),
            untextured_t2_c=lambda df: targeted_from_map(df, untextured_map_2_c),
            chicken_t2_c=lambda df: targeted_from_map(df, chicken_map_2_c),
            dairy_t2_c=lambda df: targeted_from_map(df, dairy_map_2_c)
            )

        .pipe(rolling_window_avg, 'breakfast', 'item_price', 'item_quantity', 1, 'D')
        .pipe(rolling_window_avg, 'textured', 'item_price', 'item_quantity', 1, 'D')
        .pipe(rolling_window_avg, 'untextured', 'item_price', 'item_quantity', 1, 'D')
        .pipe(rolling_window_avg, 'breakfast_t2', 'item_price', 'item_quantity', 1, 'D')
        .pipe(rolling_window_avg, 'textured_t2', 'item_price', 'item_quantity', 1, 'D')
        .pipe(rolling_window_avg, 'untextured_t2', 'item_price', 'item_quantity', 1, 'D')
        .pipe(rolling_window_avg, 'chicken_t2', 'item_price', 'item_quantity', 1, 'D')
        .pipe(rolling_window_avg, 'dairy_t2', 'item_price', 'item_quantity', 1, 'D')

        .pipe(rolling_window_avg, 'breakfast_c', 'item_price', 'item_quantity', 1, 'D')
        .pipe(rolling_window_avg, 'untextured_c', 'item_price', 'item_quantity', 1, 'D')
        .pipe(rolling_window_avg, 'breakfast_t2_c', 'item_price', 'item_quantity', 1, 'D')
        .pipe(rolling_window_avg, 'textured_t2_c', 'item_price', 'item_quantity', 1, 'D')
        .pipe(rolling_window_avg, 'untextured_t2_c', 'item_price', 'item_quantity', 1, 'D')
        .pipe(rolling_window_avg, 'chicken_t2_c', 'item_price', 'item_quantity', 1, 'D')
        .pipe(rolling_window_avg, 'dairy_t2_c', 'item_price', 'item_quantity', 1, 'D')

        .assign(
            breakfast_outcome=lambda df: 1*df['breakfast'],
            textured_outcome=lambda df: 1*df['textured'],
            untextured_outcome=lambda df: 1*df['untextured'],
            breakfast_t2_outcome=lambda df: 1*df['breakfast_t2'],
            textured_t2_outcome=lambda df: 1*df['textured_t2'],
            untextured_t2_outcome=lambda df: 1*df['untextured_t2'],
            chicken_t2_outcome=lambda df: 1*df['chicken_t2'],
            dairy_t2_outcome=lambda df: 1*df['dairy_t2'],
            
            breakfast_outcome_c=lambda df: 1*df['breakfast_c'],
            textured_outcome_c=0,
            untextured_outcome_c=lambda df: 1*df['untextured_c'],
            breakfast_t2_outcome_c=lambda df: 1*df['breakfast_t2_c'],
            textured_t2_outcome_c=lambda df: 1*df['textured_t2_c'],
            untextured_t2_outcome_c=lambda df: 1*df['untextured_t2_c'],
            chicken_t2_outcome_c=lambda df: 1*df['chicken_t2_c'],
            dairy_t2_outcome_c=lambda df: 1*df['dairy_t2_c']
            ))

In [ ]:
# Memory optimization - convert high-memory object columns to category
print(f"Memory before: {model_data.memory_usage(deep=True).sum() / 1e9:.2f} GB")

# Convert object columns that have limited unique values to category
object_cols_to_category = ['dish_category', 'item_name', 'price_point_name', 'item_modifications']
for col in object_cols_to_category:
    if col in model_data.columns and model_data[col].dtype == 'object':
        model_data[col] = model_data[col].astype('category')

# Downcast numeric columns where possible
int_cols = model_data.select_dtypes(include=['int64']).columns
for col in int_cols:
    model_data[col] = pd.to_numeric(model_data[col], downcast='integer')

float_cols = model_data.select_dtypes(include=['float64']).columns
for col in float_cols:
    model_data[col] = pd.to_numeric(model_data[col], downcast='float')

print(f"Memory after: {model_data.memory_usage(deep=True).sum() / 1e9:.2f} GB")

In [ ]:
del model_dat
gc.collect()
model_data_customers = (
    pd.merge(
        model_data.drop(columns=['_merge']).reset_index(),
        customers_before_after.explode('customer_ids'),
        left_on=['location_id','customer_id'], right_on=['loc_id','customer_ids'], how='inner', indicator=True)
    .set_index('created_at')
    # make all the targeted columns the *_c ones and delete the *_c ones
    .assign(
        breakfast_outcome=lambda df: df['breakfast_outcome_c'],
        textured_outcome=lambda df: df['textured_outcome_c'],
        untextured_outcome=lambda df: df['untextured_outcome_c'],
        breakfast_t2_outcome=lambda df: df['breakfast_t2_outcome_c'],
        textured_t2_outcome=lambda df: df['textured_t2_outcome_c'],
        untextured_t2_outcome=lambda df: df['untextured_t2_outcome_c'],
        chicken_t2_outcome=lambda df: df['chicken_t2_outcome_c'],
        dairy_t2_outcome=lambda df: df['dairy_t2_outcome_c']
        )
    .drop(columns=[col for col in model_data.columns if col.endswith('_c')])
    )

In [ ]:
# # Check for NaN in one animal category column, arranged by restaurant - show ALL unique items
# col = 'sausage'  # Pick one animal product

# print(f"NaN values in '{col}' by restaurant:\n")

# for loc_id in model_dat['location_id'].unique():
#     loc_data = model_dat[model_dat['location_id'] == loc_id]
#     if col in loc_data.columns:
#         na_count = loc_data[col].isna().sum()
#         total_count = len(loc_data)
#         if na_count > 0:
#             print(f"\n{'='*60}")
#             print(f"{loc_id}: {na_count} NaN values (out of {total_count} rows)")
#             print(f"{'='*60}")
#             na_items = loc_data[loc_data[col].isna()]['item_name'].unique()
#             print(f"All {len(na_items)} unique items with NaN:")
#             for item in sorted(na_items):
#                 print(f"  - {item}")

In [ ]:
def clip_na(group):
    valid_part = group.dropna(subset='vegan_window_avg_item_price')
    first_valid = valid_part.index.min()
    last_valid = valid_part.index.max()
    clipped = group.loc[first_valid:last_valid]
    return clipped

location_ids = [
    "VLZX7K2M9QD4T","SRQS8F7JWA9MZ","2HRX9P6HKXA8V","JHDN7CF1C03X5","L69HYJ4Y3TR91","ED5J990H5VAZT","W8T41JZK0ZMEP",
    "EMBVNVD207CC6","C0BE4NDSW26QN","75WYSXR9QBK5M","V3Q26BHF3SE2H","LBZEEFSBJNB3Z","SAFK7ND1HR6XS",
    "CB2KHY1C2G9PT","S8MT0YGD2KTN9","LFZFT3VASXPED","1SQPTEGYPH0GA","9XKJD8DQTH559","LQ5EH4BKGV61T","78AY09MVJVTYE"
    ]

def aggregate_daily_model_data(df_input):

    # Aggregate
    daily_model_data = (
        df_input
        .groupby(['location_id', df_input.index.normalize()], observed=True)
        .agg({
            
            # Price averages
            'item_quantity':'last',
            'vegan_window_sum_item_quantity':'last',
            'vegetarian_window_sum_item_quantity':'last',
            'meat_window_sum_item_quantity':'last',
            'vegan_window_avg_item_price':'last',
            'vegetarian_window_avg_item_price':'last',
            'meat_window_avg_item_price':'last',

            # Targeted price averages
            'breakfast_window_avg_item_price':'last',
            'textured_window_avg_item_price':'last',
            'untextured_window_avg_item_price':'last',

            # T2
            'breakfast_t2_window_avg_item_price':'last',
            'textured_t2_window_avg_item_price':'last',
            'untextured_t2_window_avg_item_price':'last',
            'chicken_t2_window_avg_item_price':'last',
            'dairy_t2_window_avg_item_price':'last',

            # Outcomes
            'vegan_outcome':'sum',
            'vegetarian_outcome':'sum',
            'total_outcome':'sum',
            'nonvegan_outcome':'sum',
            'meat_outcome':'sum',
            'chicken_fish_outcome':'sum',

            # Targeted outcomes
            'breakfast_outcome':'sum',
            'textured_outcome':'sum',
            'untextured_outcome':'sum',

            # T2
            'breakfast_t2_outcome':'sum',
            'textured_t2_outcome':'sum',
            'untextured_t2_outcome':'sum',
            'chicken_t2_outcome':'sum',
            'dairy_t2_outcome':'sum',

        })
        .rename(columns={'item_quantity':'item_quantity_day'})
        .rename_axis(index=['location_id','created_at'])
        .reindex(pd.MultiIndex.from_product([location_ids_by_coverage,
                                            pd.date_range(df_input.index.min().normalize(),
                                                        df_input.index.max().normalize(),
                                                        freq='D', tz='UTC')], 
                                            names=['location_id', 'created_at']))
        .reset_index()
        .set_index('created_at')
        .groupby('location_id', group_keys=False) # Switch argument in future version of Pandas
        .apply(clip_na)
        .fillna({'chicken_fish_outcome':0,
                'nonvegan_outcome':0, 
                'meat_outcome':0,
                'vegan_outcome':0, 
                'vegetarian_outcome':0,
                'total_outcome':0,
                'breakfast_outcome':0,
                'textured_outcome':0,
                'untextured_outcome':0,
                'breakfast_t2_outcome':0,
                'textured_t2_outcome':0,
                'untextured_t2_outcome':0,
                'chicken_t2_outcome':0,
                'dairy_t2_outcome':0
                })
        #.query('0 < item_quantity_day')
        .assign(
            vegan_window_avg_item_price = lambda df: df.groupby('location_id')['vegan_window_avg_item_price'].ffill(),
            vegetarian_window_avg_item_price = lambda df: df.groupby('location_id')['vegetarian_window_avg_item_price'].ffill(),
            meat_window_avg_item_price = lambda df: df.groupby('location_id')['meat_window_avg_item_price'].ffill(), 
            breakfast_window_avg_item_price = lambda df: df.groupby('location_id')['breakfast_window_avg_item_price'].ffill(),
            textured_window_avg_item_price = lambda df: df.groupby('location_id')['textured_window_avg_item_price'].ffill(),
            untextured_window_avg_item_price = lambda df: df.groupby('location_id')['untextured_window_avg_item_price'].ffill(),
            break_t2_window_avg_item_price = lambda df: df.groupby('location_id')['breakfast_t2_window_avg_item_price'].ffill(),
            textured_t2_window_avg_item_price = lambda df: df.groupby('location_id')['textured_t2_window_avg_item_price'].ffill(),
            untextured_t2_window_avg_item_price = lambda df: df.groupby('location_id')['untextured_t2_window_avg_item_price'].ffill(),
            chicken_t2_window_avg_item_price = lambda df: df.groupby('location_id')['chicken_t2_window_avg_item_price'].ffill(),
            dairy_t2_window_avg_item_price = lambda df: df.groupby('location_id')['dairy_t2_window_avg_item_price'].ffill(),
            day_of_week_cat = lambda df: df.index.to_series().dt.dayofweek.astype("category"),
            day_of_week = lambda df: df.index.to_series().dt.dayofweek.astype("category").cat.codes,
            weekend = lambda df: pd.Series(df.index.dayofweek.isin([5, 6]).astype(int), index=df.index).astype("category"),
            day_of_month_cat = lambda df: df.index.to_series().dt.day.astype("category"),
            day_of_month = lambda df: df.index.to_series().dt.day.astype("category").cat.codes,
            month_cat = lambda df: df.index.to_series().dt.month.astype("category"),
            month = lambda df: df.index.to_series().dt.month.astype("category").cat.codes,
            season = lambda df: df.index.month.map(season_from_month).astype("category"),
            year_cat = lambda df: df.index.to_series().dt.year.astype("category"),
            year = lambda df: df.index.to_series().dt.year.astype("category").cat.codes,
            date = lambda df: df.index.to_series().dt.date.astype("category").cat.codes,
            exposure_VLZX7K2M9QD4T_1 = lambda df: make_exposure(df, "VLZX7K2M9QD4T", before_after_details_true.loc["VLZX7K2M9QD4T", "cross_over_date"]),
            exposure_SRQS8F7JWA9MZ_1 = lambda df: make_exposure(df, "SRQS8F7JWA9MZ", "2020-06-25"),
            exposure_SRQS8F7JWA9MZ_2 = lambda df: make_exposure(df, "SRQS8F7JWA9MZ", before_after_details_true.loc["SRQS8F7JWA9MZ","cross_over_date"]),
            exposure_2HRX9P6HKXA8V_1 = lambda df: make_exposure(df, "2HRX9P6HKXA8V", before_after_details_true.loc["2HRX9P6HKXA8V","cross_over_date"]),
            exposure_JHDN7CF1C03X5_1 = lambda df: make_exposure(df, "JHDN7CF1C03X5", before_after_details_true.loc["JHDN7CF1C03X5","cross_over_date"]),
            exposure_JHDN7CF1C03X5_2 = lambda df: make_exposure(df, "JHDN7CF1C03X5", "2020-03-12"),
            exposure_L69HYJ4Y3TR91_1 = lambda df: make_exposure(df, "L69HYJ4Y3TR91", before_after_details_true.loc["L69HYJ4Y3TR91","cross_over_date"]),
            exposure_ED5J990H5VAZT_1 = lambda df: make_exposure(df, "ED5J990H5VAZT", before_after_details_true.loc["ED5J990H5VAZT","cross_over_date"]),
            exposure_W8T41JZK0ZMEP_1 = lambda df: make_exposure(df, "W8T41JZK0ZMEP", before_after_details_true.loc["W8T41JZK0ZMEP","cross_over_date"]),
            exposure_W8T41JZK0ZMEP_2 = lambda df: make_exposure(df, "W8T41JZK0ZMEP", "2020-04-27"),
            exposure_W8T41JZK0ZMEP_3 = lambda df: make_exposure(df, "W8T41JZK0ZMEP", "2020-09-29"),
            exposure_W8T41JZK0ZMEP_4 = lambda df: make_exposure(df, "W8T41JZK0ZMEP", "2020-07-13"),
            exposure_W8T41JZK0ZMEP_5 = lambda df: make_exposure(df, "W8T41JZK0ZMEP", "2021-02-04"),
            exposure_EMBVNVD207CC6_1 = lambda df: make_exposure(df, "EMBVNVD207CC6", before_after_details_true.loc["EMBVNVD207CC6","cross_over_date"]),
            exposure_C0BE4NDSW26QN_1 = lambda df: make_exposure(df, "C0BE4NDSW26QN", before_after_details_true.loc["C0BE4NDSW26QN","cross_over_date"]),
            exposure_C0BE4NDSW26QN_2 = lambda df: make_exposure(df, "C0BE4NDSW26QN", "2019-09-03"),
            exposure_75WYSXR9QBK5M_1 = lambda df: make_exposure(df, "75WYSXR9QBK5M", before_after_details_true.loc["75WYSXR9QBK5M","cross_over_date"]),
            exposure_V3Q26BHF3SE2H_1 = lambda df: make_exposure(df, "V3Q26BHF3SE2H", before_after_details_true.loc["V3Q26BHF3SE2H","cross_over_date"]),
            exposure_V3Q26BHF3SE2H_2 = lambda df: make_exposure(df, "V3Q26BHF3SE2H", "2021-03-06"),
            exposure_V3Q26BHF3SE2H_3 = lambda df: make_exposure(df, "V3Q26BHF3SE2H", "2021-09-18"),
            exposure_V3Q26BHF3SE2H_4 = lambda df: make_exposure(df, "V3Q26BHF3SE2H", "2021-06-11"),
            exposure_LBZEEFSBJNB3Z_1 = lambda df: make_exposure(df, "LBZEEFSBJNB3Z", before_after_details_true.loc["LBZEEFSBJNB3Z","cross_over_date"]),
            exposure_SAFK7ND1HR6XS_1 = lambda df: make_exposure(df, "SAFK7ND1HR6XS", before_after_details_true.loc["SAFK7ND1HR6XS","cross_over_date"]),
            exposure_CB2KHY1C2G9PT_1 = lambda df: make_exposure(df, "CB2KHY1C2G9PT", before_after_details_true.loc["CB2KHY1C2G9PT","cross_over_date"]),
            exposure_S8MT0YGD2KTN9_1 = lambda df: make_exposure(df, "S8MT0YGD2KTN9", before_after_details_true.loc["S8MT0YGD2KTN9","cross_over_date"]),
            exposure_LFZFT3VASXPED_1 = lambda df: make_exposure(df, "LFZFT3VASXPED", before_after_details_true.loc["LFZFT3VASXPED","cross_over_date"]),
            exposure_1SQPTEGYPH0GA_1 = lambda df: make_exposure(df, "1SQPTEGYPH0GA", before_after_details_true.loc["1SQPTEGYPH0GA","cross_over_date"]),
            exposure_9XKJD8DQTH559_1 = lambda df: make_exposure(df, "9XKJD8DQTH559", before_after_details_true.loc["9XKJD8DQTH559","cross_over_date"]),
            exposure_9XKJD8DQTH559_2 = lambda df: make_exposure(df, "9XKJD8DQTH559", "2021-03-26"),
            exposure_LQ5EH4BKGV61T_1 = lambda df: make_exposure(df, "LQ5EH4BKGV61T", before_after_details_true.loc["LQ5EH4BKGV61T","cross_over_date"]),
            exposure_78AY09MVJVTYE_1 = lambda df: make_exposure(df, "78AY09MVJVTYE", before_after_details_true.loc["78AY09MVJVTYE","cross_over_date"]),
            )
        )
    
    return daily_model_data

# Aggregate
#daily_model_data = aggregate_daily_model_data(model_data)
daily_model_data_customers = aggregate_daily_model_data(model_data_customers)

# # Export
# filename =  DATA_DIR_4 / 'all_locations_daily.parquet'
# if not filename.exists():
#     daily_model_data = pd.merge(daily_model_data, locations, left_on='location_id', right_index=True, how='left')

filename_customer = DATA_DIR_4 / 'customer' / 'all_locations_daily_customers.parquet'
if not filename_customer.exists():
   daily_model_data_customers = pd.merge(daily_model_data_customers, locations, left_on='location_id', right_index=True, how='left')

In [ ]:
dish_counts_list = []
for loc_id in location_ids_by_coverage[1:]:
    dish_counts = (
        pd.read_csv(Path('scripts') / 'labeling' / 'dish_counts' / f'{loc_id}.csv',index_col=0)
        .rename_axis('created_at')
        .reset_index()
        .assign(
            location_id = loc_id, 
            created_at = lambda df: pd.to_datetime(df.created_at, utc=True).dt.normalize(),
            breakfast_dishes_count = lambda df: df.sausage_dishes_count + df.bacon_dishes_count + df.breakfast_sausage_patty_dishes_count,
            textured_dishes_count = lambda df: df.lamb_dishes_count + df.chunked_beef_or_pork_dishes_count + df.pulled_pork_dishes_count,
            untextured_dishes_count = lambda df: df.beef_or_pork_burger_dishes_count + df.ground_meat_dishes_count + df.meatballs_dishes_count,
            breakfast_dishes_presence = lambda df: (df.breakfast_dishes_count > 0).astype(int),
            textured_dishes_presence = lambda df: (df.textured_dishes_count > 0).astype(int),
            untextured_dishes_presence = lambda df: (df.untextured_dishes_count > 0).astype(int),
        )
        .pipe(lambda df: df.assign(**{
            f"{c.replace('_count','')}_prop": (df[c] / df["dishes_count"]).fillna(0)
            for c in df.columns
            if c == "vegan_dishes_count" or c == "vegetarian_dishes_count" or c == "mpbamod_dishes_count"
        })))
    dish_counts_list.append(dish_counts)
dish_count_df = pd.concat(dish_counts_list, ignore_index=True)

cols_to_ffill = [c for c in dish_count_df.columns if c not in ['location_id','created_at']]

# daily_model = (
#     pd.merge(daily_model_data.reset_index(), dish_count_df, on=['location_id','created_at'], how='left')
#     # ffill all new columns per each restaurant, only new columns and not old ones
#     .groupby('location_id', group_keys=False)
#     .apply(lambda g: g.assign(**{c: g[c].ffill().bfill().fillna(0) for c in cols_to_ffill}))
# )
daily_model_customer = (
    pd.merge(daily_model_data_customers.reset_index(), dish_count_df, on=['location_id','created_at'], how='left')
    .groupby('location_id', group_keys=False)
    .apply(lambda g: g.assign(**{c: g[c].ffill().bfill().fillna(0) for c in cols_to_ffill}))
)
#daily_model.to_parquet(DATA_DIR_4 / 'all_locations_daily.parquet')
daily_model_customer.to_parquet(DATA_DIR_4 / 'customer' / 'all_locations_daily_customers.parquet')

In [ ]:
for dishes_count_col in [
    'vegan_dishes_count','vegetarian_dishes_count','mpbamod_dishes_count', 
    'vegan_dishes_prop','vegetarian_dishes_prop','mpbamod_dishes_prop']:

    (
        daily_model
        .assign(location_id_copy = lambda df: df.location_id)
        .set_index(['created_at','location_id'])
        .filter(regex=r'^(?!.*exposure)', axis=1)
        .pipe(lambda df: df.join(
            df
            .pivot_table(
                index=['created_at','location_id'],
                columns='location_id_copy',
                values=dishes_count_col,
                fill_value=0)
            .pipe(lambda df: df.rename(columns=lambda c: f"exposure_{c}_1"))))
        .reset_index(level='location_id')
        .to_parquet(DATA_DIR_4 / 'proportion' / f'all_locations_daily_{dishes_count_col}.parquet')
    )


for dishes_count_col in [
    'breakfast_dishes_count','textured_dishes_count','untextured_dishes_count',#'chicken_dishes_count','dairy_dishes_count',
    'breakfast_dishes_presence','textured_dishes_presence','untextured_dishes_presence'#,'chicken_dishes_presence','dairy_dishes_presence'
    ]:

    (
        daily_model
        .assign(location_id_copy = lambda df: df.location_id)
        .set_index(['created_at','location_id'])
        .filter(regex=r'^(?!.*exposure)', axis=1)
        .pipe(lambda df: df.join(
            df
            .pivot_table(
                index=['created_at','location_id'],
                columns='location_id_copy',
                values=dishes_count_col,
                fill_value=0)
            .pipe(lambda df: df.rename(columns=lambda c: f"exposure_{c}_1"))))
        .reset_index(level='location_id')
        .to_parquet(DATA_DIR_4 / 'proportion_targeted' / f'all_locations_daily_{dishes_count_col}.parquet')
    )

    

Plotting tool

In [ ]:
# # Plot function for resampling and visualization
# def plot_resampled(data, freq, start_date=None, end_date=None, title="Resampled Predictions"):
#     resampled_pred = data.resample(freq)['pred'].mean()
#     resampled_actual = data.resample(freq)['vegan_outcome'].mean()

#     if start_date and end_date:
#         resampled_pred = resampled_pred.loc[start_date:end_date]
#         resampled_actual = resampled_actual.loc[start_date:end_date]

#     resampled_pred.plot(color='orange', label='Predicted', title=title)
#     resampled_actual.plot(color='blue', alpha=0.3, label='Actual', title=title)

In [ ]:
# before_after_customers_by_loc = (pd.read_pickle('data/before_after_customers.pkl')
#                                  .set_index('loc_id')
#                                  ['customer_ids']
#                                  .loc[location_ids_by_coverage[1:]])

# ncols = 4
# nrows = math.ceil(len(before_after_customers_by_loc) / ncols)
# fig, axes = plt.subplots(nrows, ncols, figsize=(20, 5 * nrows))
# axes = axes.flatten()
# for i, (loc_id, customers_in_loc) in enumerate(before_after_customers_by_loc.items()):
    
#     ax = axes[i]
#     if not customers_in_loc:
#         ax.set_title(f"Location {loc_id}\nNo customers found.")
#         ax.axis('off') # Turn off the axis if no data.
#         continue

#     customer_orders_day = (model_data
#                            .assign(temp_date=lambda df: df.index.date)
#                            .query('customer_id.isin(@customers_in_loc)')
#                            .groupby(['customer_id', 'temp_date'])
#                            ['nonvegan_outcome']
#                            .sum())

#     if customer_orders_day.empty:
#         ax.set_title(f"Location {loc_id}\nNo order data found.")
#         ax.axis('off') # Turn off the axis if no data.
#         continue

#     customer_mean_day = customer_orders_day.groupby('customer_id').transform('mean').round().astype(int)
#     deviations = customer_orders_day.sub(customer_mean_day)
#     deviation_counts = deviations.value_counts().sort_index()

#     ax.bar(deviation_counts.index, deviation_counts.values, color='skyblue', edgecolor='black')
#     ax.set_title(f'Location ID: {loc_id}')
#     ax.set_xlabel('Deviations from Mean')
#     ax.set_ylabel('Frequency')
#     ax.set_xlim(-20.5, 20.5)
#     ax.set_xticks(ticks=range(-20, 21, 5)) # Adjusted ticks for smaller subplot size
#     ax.grid(axis='y', linestyle='--', alpha=0.7)

# for j in range(i + 1, len(axes)):
#     axes[j].axis('off')
# fig.suptitle('Customer Orders Day Deviation from Their Means by Location', fontsize=24, fontweight='bold')
# plt.tight_layout(rect=[0, 0.03, 1, 0.96])
# plt.show()
# location_ids_by_coverage.remove('VLZX7K2M9QD4T')
# before_after_customers_by_loc = (pd.read_pickle('data/before_after_customers.pkl')
#                                    .set_index('loc_id')
#                                    ['customer_ids']
#                                    .loc[location_ids_by_coverage])
# location_ids_by_coverage.insert(0, 'VLZX7K2M9QD4T')

# ncols = 4
# nrows = math.ceil(len(before_after_customers_by_loc) / ncols)
# fig, axes = plt.subplots(nrows, ncols, figsize=(20, 6 * nrows))
# axes = axes.flatten()

# # --- MODIFICATION: Create a colormap for the 1-10 purchase range ---
# min_purchases = 1
# max_purchases = 8
# num_colors = max_purchases - min_purchases + 1
# colors = plt.cm.viridis(np.linspace(0, 1, num_colors))

# for i, (loc_id, customers_in_loc) in enumerate(before_after_customers_by_loc.items()):
    
#     ax = axes[i]
#     if not customers_in_loc:
#         ax.set_title(f"Location {loc_id}\nNo customers found.")
#         ax.axis('off')
#         continue

#     customer_orders_day = (model_data
#                            .assign(temp_date=lambda df: df.index.date)
#                            .query('customer_id.isin(@customers_in_loc)')
#                            .groupby(['customer_id', 'temp_date'])
#                            ['nonvegan_outcome']
#                            .sum())

#     if customer_orders_day.empty:
#         ax.set_title(f"Location {loc_id}\nNo order data found.")
#         ax.axis('off')
#         continue
    
#     # Prepare data for stacked bar chart
#     deviation_df = pd.DataFrame({'orders': customer_orders_day})
#     deviation_df['mean'] = deviation_df.groupby('customer_id')['orders'].transform('mean').round().astype(int)
#     deviation_df['deviation'] = deviation_df['orders'] - deviation_df['mean']
    
#     # Group by both deviation and the original number of orders to get the counts for each segment.
#     stacked_data = deviation_df.groupby(['deviation', 'orders']).size().unstack(fill_value=0)

#     # Create the stacked bar plot
#     bottom = np.zeros(len(stacked_data))

#     # Loop through each original order count to create the stacks.
#     for order_count, counts_per_deviation in stacked_data.items():
#         # --- MODIFICATION: Only plot bars for purchase counts between 1 and 10 ---
#         if min_purchases <= order_count <= max_purchases:
#             # Map the order count (1-10) to a color index (0-9)
#             color_index = order_count - min_purchases
#             color = colors[color_index]
#             ax.bar(stacked_data.index, counts_per_deviation, bottom=bottom, label=f'{order_count} orders', color=color, edgecolor='white', linewidth=0.7)
#             bottom += counts_per_deviation.values

#     ax.set_title(f'Location ID: {loc_id}')
#     ax.set_xlabel('Deviations from Mean')
#     ax.set_ylabel('Frequency')
#     ax.set_xlim(-15.5, 15.5)
#     ax.set_xticks(ticks=range(-15, 16, 5))
#     ax.grid(axis='y', linestyle='--', alpha=0.3)

# # Add a single, shared legend for the entire figure
# handles, labels = ax.get_legend_handles_labels()
# if handles:
#     fig.legend(handles, labels, title='Original Purchases', loc='center right', bbox_to_anchor=(1.05, 0.5))

# for j in range(i + 1, len(axes)):
#     axes[j].axis('off')

# fig.suptitle('Customer Orders Day Deviation from Their Means by Location', fontsize=24, fontweight='bold')
# plt.tight_layout(rect=[0, 0.03, 0.95, 0.96])
# plt.show()
# location_ids_by_coverage.remove('VLZX7K2M9QD4T')

# before_after_customers_by_loc = (pd.read_pickle('data/before_after_customers.pkl')
#                                    .set_index('loc_id')
#                                    ['customer_ids']
#                                    .loc[location_ids_by_coverage])

# location_ids_by_coverage.insert(0, 'VLZX7K2M9QD4T')

# ncols = 4
# nrows = math.ceil(len(before_after_customers_by_loc) / ncols)
# fig, axes = plt.subplots(nrows, ncols, figsize=(20, 6 * nrows))
# axes = axes.flatten()

# # --- MODIFICATION: Define colors for gender ---
# gender_colors = {'male': 'cornflowerblue', 'female': 'lightcoral'}

# for i, (loc_id, customers_in_loc) in enumerate(before_after_customers_by_loc.items()):
    
#     ax = axes[i]
#     if not customers_in_loc:
#         ax.set_title(f"Location {loc_id}\nNo customers found.")
#         ax.axis('off')
#         continue

#     customer_orders_day = (model_data
#                            .assign(temp_date=lambda df: df.index.date)
#                            .query('customer_id.isin(@customers_in_loc)')
#                            .groupby(['customer_id', 'temp_date'])
#                            ['nonvegan_outcome']
#                            .sum())

#     if customer_orders_day.empty:
#         ax.set_title(f"Location {loc_id}\nNo order data found.")
#         ax.axis('off')
#         continue
    
#     # --- MODIFICATION: Prepare data for gender-stacked bar chart ---
#     # Convert series to DataFrame and reset index to get customer_id as a column.
#     deviation_df = customer_orders_day.to_frame(name='orders').reset_index()
    
#     # Merge with the customers DataFrame to get gender information.
#     # Assuming 'customers' DataFrame has 'customer_id' and 'gender' columns.
#     deviation_df = pd.merge(deviation_df, customers, on='customer_id', how='left')

#     # Calculate mean and deviation after merging.
#     deviation_df['mean'] = deviation_df.groupby('customer_id')['orders'].transform('mean').round().astype(int)
#     deviation_df['deviation'] = deviation_df['orders'] - deviation_df['mean']
    
#     # Group by deviation and gender to get counts for stacking.
#     stacked_data = deviation_df.groupby(['deviation', 'gender']).size().unstack(fill_value=0)
    
#     # Ensure both Male and female columns exist to avoid errors.
#     if 'male' not in stacked_data: stacked_data['male'] = 0
#     if 'female' not in stacked_data: stacked_data['female'] = 0

#     # --- MODIFICATION: Create the stacked bar plot ---
#     # Plot Male bars first (the bottom layer).
#     ax.bar(stacked_data.index, stacked_data['male'], color=gender_colors['male'], label='male', edgecolor='white')
#     # Plot female bars on top of the Male bars.
#     ax.bar(stacked_data.index, stacked_data['female'], bottom=stacked_data['male'], color=gender_colors['female'], label='female', edgecolor='white')

#     ax.set_title(f'Location ID: {loc_id}')
#     ax.set_xlabel('Deviations from Mean')
#     ax.set_ylabel('Frequency')
#     ax.set_xlim(-20.5, 20.5)
#     ax.set_xticks(ticks=range(-20, 21, 5))
#     ax.grid(axis='y', linestyle='--', alpha=0.7)

# # --- Add a single, shared legend for the entire figure ---
# handles = [plt.Rectangle((0,0),1,1, color=gender_colors[label]) for label in ['male', 'female']]
# labels = ['male', 'female']
# fig.legend(handles, labels, title='Gender', loc='center right', bbox_to_anchor=(1.05, 0.5))

# for j in range(i + 1, len(axes)):
#     axes[j].axis('off')

# fig.suptitle('Customer Orders Day Deviation by Gender', fontsize=24, fontweight='bold')
# plt.tight_layout(rect=[0, 0.03, 0.95, 0.96])
# plt.show()